In [1]:
import pandas as pd

df = pd.read_csv("../data/processed/dataco_cleaned.csv")

order_columns = [
    "order_date",
    "shipping_date",
    "delivery_status",
    "late_delivery_risk",
    "actual_shipping_days",
    "scheduled_shipping_days",
    "shipping_mode",
    "market",
    "order_region",
    "order_country",
    "order_state",
    "order_city",
    "order_status"
]

# Check whether the same order has different values
# in fields that should belong to the order itself.
inconsistent_orders = (
    df.groupby("order_id")[order_columns]
    .nunique(dropna=False)
    .gt(1)
    .any(axis=1)
)

print(
    "Orders with inconsistent attributes:",
    inconsistent_orders.sum()
)

print(
    "Total unique orders:",
    df["order_id"].nunique()
)


Orders with inconsistent attributes: 0
Total unique orders: 65752


Imagine an order with three items. We only want one record for order 1001 in our orders table, but all three records in order_items

In [2]:
%pip install mysql-connector-python

Note: you may need to restart the kernel to use updated packages.


In [3]:
import mysql.connector
from getpass import getpass

password = getpass("Enter MySQL password: ")

connection = mysql.connector.connect(
    host="localhost",
    user="root",
    password=password,
    database="logistics_kpi"
)

if connection.is_connected():
    print("Successfully connected to MySQL!")


Enter MySQL password: ········
Successfully connected to MySQL!


In [4]:
cursor = connection.cursor()

cursor.execute("SHOW TABLES;")

for table in cursor.fetchall():
    print(table)

('order_items',)
('orders',)


Check whether your tables are empty

In [5]:
cursor = connection.cursor()

for table in ["orders", "order_items"]:
    cursor.execute(f"SELECT COUNT(*) FROM {table}")
    print(f"{table}: {cursor.fetchone()[0]:,} rows")

cursor.close()

orders: 0 rows
order_items: 0 rows


True

In [6]:
import pandas as pd

df = pd.read_csv(
    "../data/processed/dataco_cleaned.csv",
    parse_dates=["order_date", "shipping_date"]
)

orders_df = (
    df.drop(columns=["order_item_id", "quantity", "sales"])
      .drop_duplicates(subset=["order_id"])
      .copy()
)

order_items_df = df[
    ["order_item_id", "order_id", "quantity", "sales"]
].copy()

print("Orders:", len(orders_df))
print("Order items:", len(order_items_df))


Orders: 65752
Order items: 180519


Import orders

In [7]:
from datetime import datetime
import pandas as pd

order_columns = [
    "order_id", "order_date", "shipping_date",
    "delivery_status", "late_delivery_risk",
    "actual_shipping_days", "scheduled_shipping_days",
    "shipping_mode", "market", "order_region",
    "order_country", "order_state", "order_city",
    "order_status"
]

placeholders = ", ".join(["%s"] * len(order_columns))
column_names = ", ".join(order_columns)

insert_orders_sql = f"""
INSERT INTO orders ({column_names})
VALUES ({placeholders})
"""

def to_mysql_records(dataframe):
    records = []
    for row in dataframe.itertuples(index=False, name=None):
        records.append(tuple(
            None if pd.isna(value)
            else value.to_pydatetime() if isinstance(value, pd.Timestamp)
            else value.item() if hasattr(value, "item")
            else value
            for value in row
        ))
    return records

cursor = connection.cursor()

try:
    for start in range(0, len(orders_df), 5000):
        batch = orders_df[order_columns].iloc[start:start + 5000]
        cursor.executemany(
            insert_orders_sql,
            to_mysql_records(batch)
        )

    connection.commit()
    print("Orders imported successfully!")

except Exception:
    connection.rollback()
    raise

finally:
    cursor.close()


Orders imported successfully!


Import order items

In [8]:
insert_items_sql = """
INSERT INTO order_items (
    order_item_id,
    order_id,
    quantity,
    sales
)
VALUES (%s, %s, %s, %s)
"""

cursor = connection.cursor()

try:
    for start in range(0, len(order_items_df), 5000):
        batch = order_items_df.iloc[start:start + 5000]

        cursor.executemany(
            insert_items_sql,
            to_mysql_records(batch)
        )

    connection.commit()
    print("Order items imported successfully!")

except Exception:
    connection.rollback()
    raise

finally:
    cursor.close()


Order items imported successfully!


Verify the imported data

In [9]:
cursor = connection.cursor()

cursor.execute("SELECT COUNT(*) FROM orders")
total_orders = cursor.fetchone()[0]

cursor.execute("SELECT COUNT(*) FROM order_items")
total_items = cursor.fetchone()[0]

print(f"Total orders in MySQL: {total_orders:,}")
print(f"Total order items in MySQL: {total_items:,}")

cursor.close()


Total orders in MySQL: 65,752
Total order items in MySQL: 180,519


True

In [10]:
%pip install pandas mysql-connector-python openpyxl python-dotenv

Note: you may need to restart the kernel to use updated packages.
